# 01 · Data & EDA — *Train on what you test?*

**Direction 01, the loss-function study.** This notebook is the data stage: how MUSDB18 is acquired and licensed, the frozen 86/14/50 split, and the exploratory analysis whose findings *motivate* the five losses we compare.

It is written to be read top-to-bottom before anything runs — **no cell here has been executed.** Cells that need the dataset or the network are flagged with a bold ⚠️ RUN THIS LATER banner and a time estimate.

### State of this notebook
- **Contract:** [`../MASTER_PLAN.md`](../MASTER_PLAN.md) §4 (data), §14 (notebook spec).
- **Splits:** §4.2 — 86 train / 14 valid / 50 test, names in `../configs/splits.csv`.
- **Why EDA here:** each statistic ties to a later design choice (log-domain losses, the SI-SDR silence guard), never decoration.
- **Nothing is trained.** All GPU work is deferred to notebook 03.

In [ ]:
# === Colab bootstrap — RUN THIS LATER (Colab only) ==========================
# ⚠️ RUN THIS LATER · ~2 min · CPU. Installs the pinned env and mounts Drive.
# Skip locally if you already `pip install -r requirements.txt`.
#
# !git clone https://github.com/SeanSalvador2/vocal-separation-research.git
# %cd vocal-separation-research
# !pip install -r requirements.txt
# from google.colab import drive; drive.mount('/content/drive')
# import os; os.environ["SHARD_ROOT"] = "/content/drive/MyDrive/musdb_shards"
print("Bootstrap cell — run on Colab only (see comments). No-op here.")


## 1 · License & acquisition
MUSDB18 (Zenodo record **1117372**, 4.7 GB, 150 stereo tracks @ 44.1 kHz) is **research/education only and non-redistributable** — *no audio is ever committed*; it lives on Drive/local scratch. We use the compressed STEMS distribution and decode it **once** to per-track WAV shards.

> ⚠️ **RUN THIS LATER** — decode STEMS → WAV shards (CPU + network)  ·  _~30–60 min once · CPU_

In [ ]:
# ⚠️ RUN THIS LATER (see banner above). Decodes once; do not re-run per session.
# !python scripts/prepare_data.py --musdb-root $MUSDB_ROOT --out $SHARD_ROOT
# !python scripts/prepare_data.py --write-manifest --out $SHARD_ROOT \
#     --splits-csv 01-loss-function-study/configs/splits.csv
# !python scripts/prepare_data.py --verify --out $SHARD_ROOT
print('Data prep is RUN LATER — needs the MUSDB18 archive on disk.')

## 2 · The frozen split manifest
The 14-track **validation** split is the exact `musdb` `split="valid"` list (verified verbatim from the sigsep source; the same split Open-Unmix uses). The committed `splits.csv` carries those 14 real names plus documented placeholders for the 86 train / 50 test rows; `prepare_data.py --write-manifest` fills the real names from the decoded dataset. The cell below reads the committed CSV and needs no data — it runs on CPU in a blink.

In [ ]:
from singnet.data import Manifest
manifest = Manifest.from_csv('01-loss-function-study/configs/splits.csv')
print('split counts:', manifest.counts)          # {'train':86,'valid':14,'test':50}
print('validation tracks:')
for name in manifest.tracks_for('valid'):
    print('  ', name)

## 3 · EDA that drives the loss choice
Each analysis below maps to a decision in the study. All read the decoded shards, so they are RUN LATER (fast on CPU once the data exists).

### 3.1 Track duration & loudness
Sanity-checks the corpus (≈10 h) and gives per-track loudness for the listening-kit level-matching later.

> ⚠️ **RUN THIS LATER** — per-track duration/LUFS table from shards  ·  _~1–2 min · CPU_

In [ ]:
# ⚠️ RUN THIS LATER. Reads $SHARD_ROOT; builds a per-track duration/loudness table.
# import json, pandas as pd
# index = json.load(open(f'{SHARD_ROOT}/index.json'))
# df = pd.DataFrame(index).T   # duration_s, n_samples, subset
# display(df.describe())
print('RUN LATER — needs decoded shards.')

### 3.2 Vocal-activity ratio per track
Fraction of frames whose vocal RMS clears −60 dBFS. **Why it matters:** it sets expectations for the SI-SDR silence guard (§4.4) and is the baseline the uniform-random chunk sampler is deliberately *not* activity-weighting (that question belongs to Direction 08). Read the histogram as: mass near 1.0 = vocal-dense tracks; a left tail = tracks with long instrumental passages that will trigger the guard.

In [ ]:
# ⚠️ RUN THIS LATER (~2 min · CPU). Per-track vocal-activity ratio + histogram.
# import numpy as np, soundfile as sf, matplotlib.pyplot as plt
# thr = 10 ** (-60/20)
# ratios = {}
# for track in manifest.tracks_for('train'):
#     v, sr = sf.read(f'{SHARD_ROOT}/{track}/vocals.wav')
#     frames = np.array_split(np.abs(v).mean(axis=-1), max(1, len(v)//sr))
#     ratios[track] = float(np.mean([f.mean() > thr for f in frames]))
# plt.hist(list(ratios.values()), bins=20); plt.xlabel('vocal-activity ratio')
print('RUN LATER — needs decoded shards.')

### 3.3 Magnitude vs log-magnitude distributions
Histograms of $|S|$ and $\log(1+|S|)$ over vocal spectrogram bins. **Why it matters:** magnitude is heavy-tailed (a few loud bins dominate), which is exactly the regime where a **log-domain loss** (`logl1mag`, and the log term inside `l1mrstft`) re-weights quiet structure. Read the two panels as: raw magnitude piles at ~0 with a long tail; the log transform spreads that mass into a usable range — visual motivation for testing log losses.

In [ ]:
# ⚠️ RUN THIS LATER (~2 min · CPU). Magnitude & log-magnitude histograms.
# import torch, matplotlib.pyplot as plt
# from singnet.audio import STFT, drop_nyquist
# stft = STFT()
# mag = drop_nyquist(stft.transform(torch.from_numpy(v).float().mean(-1)).abs())
# fig, ax = plt.subplots(1, 2)
# ax[0].hist(mag.flatten().numpy(), bins=100); ax[0].set_title('|S|')
# ax[1].hist(mag.log1p().flatten().numpy(), bins=100); ax[1].set_title('log(1+|S|)')
print('RUN LATER — needs decoded shards.')

### 3.4 Silent-region statistics (feeds the SI-SDR guard & Direction 08)
Distribution of per-chunk vocal RMS across the train split, with the −60 dBFS line drawn. **Why it matters:** the area left of that line is the fraction of chunks the `sisdr` arm will skip — pre-registered to be *reported* (if >20 %, the finding reframes as 'SI-SDR loss is fragile on real music', §12/§15). This is the empirical basis for the guard threshold.

In [ ]:
# ⚠️ RUN THIS LATER (~2 min · CPU). Per-chunk vocal-RMS histogram vs -60 dBFS.
# from singnet.losses.sisdr import SILENCE_RMS_THRESHOLD
# ... compute 6 s chunk RMS over the train split, plt.axvline(SILENCE_RMS_THRESHOLD)
print('SILENCE threshold =', 'RUN LATER — imports singnet.losses.sisdr')

### 3.5 Spectrogram gallery
Mixture vs isolated-vocals log-magnitude spectrograms for a few tracks. **How to read:** the vocal panel shows the harmonic comb + formant bands the mask must recover; comparing to the mixture panel shows how much accompaniment overlaps them (the separation difficulty).

In [ ]:
# ⚠️ RUN THIS LATER (~1 min · CPU). Log-mag spectrogram gallery (mixture vs vocals).
print('RUN LATER — needs decoded shards.')

## 4 · Interpretation stubs (fill after running §3)
- **If magnitude is strongly heavy-tailed** → log-domain losses are well motivated; expect `logl1mag`/`l1mrstft` to shape quiet bins differently from `l1mag`/`msemag`.
- **If the silent-chunk fraction is small (<20 %)** → the SI-SDR guard is a minor correction and `sisdr` is a fair comparison; **if large** → pre-register the 'SI-SDR is fragile on music' reading (feeds Direction 08).
- **If vocal activity varies widely across tracks** → note it as a source of per-track SI-SDR variance in the §7 error analysis.